## 1. Install libraries
Internet must be ON for this notebook (pretrained weights are downloaded).

In [ ]:
!pip install -q segmentation-models-pytorch timm
!pip download segmentation-models-pytorch --no-deps -d /kaggle/working/smp_wheels -q

In [ ]:
import segmentation_models_pytorch as smp
import timm, torch
print('smp', smp.__version__, '| timm', timm.__version__, '| torch', torch.__version__)

## 2. Weights & Biases login

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_ENTITY = '23f1003053-dl-genai-project'
WANDB_PROJECT = '23f1003053-t32026'
WANDB_MODE = 'online'

try:
    wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
except Exception as e:
    WANDB_MODE = 'disabled'
    print('W&B login failed, logging disabled:', repr(e)[:150])

## 3. Configuration
Every hyper-parameter lives in one dictionary, so each W&B run records exactly what produced it.
Set `quick=True` for a 2-epoch smoke test (tiny subset) before the real run.

In [ ]:
CFG = dict(
    arch='Unet',
    encoder='tu-efficientnet_b3',          # timm EfficientNet-B3 through smp
    fallback_encoders=['resnet50', 'resnet34'],
    epochs=40, batch=16,
    lr_encoder=2e-4, lr_decoder=1e-3, weight_decay=1e-4,
    warmup_epochs=2, min_lr_factor=0.01,
    grad_clip=5.0,
    ema_decay=0.999,
    cls_weight=0.5,                        # weight of the classification-head loss
    patience=8,                            # early stopping on val official score
    class_sample_weight={'defect_a': 2.0, 'defect_b': 4.0, 'defect_c': 1.5, 'defect_d': 2.0},
    seed=42, val_size=0.2,
    quick=False, resume=False,
)
RUN_NAME = 'unet_effb3_clshead_ema'

## 4. Data, split and sampling weights
Identical split to the baseline (stratified 80/20 on the *combination* of defects present, seed 42), so validation scores can be compared one-to-one.

In [ ]:
import os, time, random, json, gc, copy, math
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split

H, W = 128, 800
N = H * W
CLASSES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
SEED = CFG['seed']
EPOCHS = CFG['epochs']
BATCH = CFG['batch']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = device == 'cuda'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True


def find_file(name):
    for root, _, files in os.walk('/kaggle/input'):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)


df = pd.read_csv(find_file('train.csv'))
df['mask_rle'] = df['mask_rle'].fillna('').astype(str)
df['has'] = df['mask_rle'].str.strip() != ''
pos = df[df['has']]

img_paths = {}
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
            img_paths[os.path.splitext(f)[0]] = os.path.join(root, f)

labels = pos.groupby('image_id')['defect_type'].apply(lambda s: '+'.join(sorted(s)))
combo = labels.reindex(df['image_id'].drop_duplicates()).fillna('clean')
rle_map = {(r.image_id, r.defect_type): r.mask_rle for r in df.itertuples()}
train_ids, val_ids = train_test_split(combo.index.to_numpy(), test_size=CFG['val_size'], stratify=combo.values, random_state=SEED)

if CFG['quick']:
    train_ids, val_ids, EPOCHS = train_ids[:320], val_ids[:96], 2

# sampling weight of an image = weight of the rarest-class defect it contains (clean images = 1.0)
combo_d = combo.to_dict()
def image_weight(iid):
    c = combo_d[iid]
    return 1.0 if c == 'clean' else max(CFG['class_sample_weight'][x] for x in c.split('+'))
train_w = np.array([image_weight(i) for i in train_ids], dtype=np.float64)

print('device:', device, '| images found:', len(img_paths))
print('train:', len(train_ids), '| val:', len(val_ids), '| epochs:', EPOCHS)
print('clean share in train (natural):', round(float((combo[train_ids] == 'clean').mean()), 3),
      '| expected clean share after weighted sampling:', round(float(train_w[(combo[train_ids] == 'clean').values].sum() / train_w.sum()), 3))

## 5. Dataset with augmentation
Augmentations are written by hand with OpenCV (no extra library, no version surprises) and are applied **identically to the image and its 4 masks**:
- horizontal / vertical flip
- brightness + contrast jitter (steel strips differ in lighting)
- small shift + scale (+/-5% shift, +/-10% scale)
- light Gaussian noise
No large rotations: defect orientation (e.g. long scratches) is informative.

In [ ]:
def rle_decode(rle):
    mask = np.zeros(N, dtype=np.uint8)
    if rle.strip():
        t = list(map(int, rle.split()))
        for start, length in zip(t[0::2], t[1::2]):
            a = start - 1
            mask[a:min(a + length, N)] = 1
    return mask.reshape((H, W), order='F')


def augment(img, masks):
    # img: (H, W) float32 in [0, 1]; masks: (4, H, W) float32 in {0, 1}
    if random.random() < 0.5:
        img, masks = img[:, ::-1], masks[:, :, ::-1]
    if random.random() < 0.5:
        img, masks = img[::-1], masks[:, ::-1]
    img = np.ascontiguousarray(img); masks = np.ascontiguousarray(masks)
    if random.random() < 0.5:
        img = np.clip(img * random.uniform(0.8, 1.2) + random.uniform(-0.1, 0.1), 0, 1)
    if random.random() < 0.5:
        s = random.uniform(0.9, 1.1)
        tx, ty = random.uniform(-0.05, 0.05) * W, random.uniform(-0.05, 0.05) * H
        M = np.array([[s, 0, (1 - s) * W / 2 + tx], [0, s, (1 - s) * H / 2 + ty]], dtype=np.float32)
        img = cv2.warpAffine(img.astype(np.float32), M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
        masks = np.stack([cv2.warpAffine(m, M, (W, H), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT_101) for m in masks])
    if random.random() < 0.2:
        img = img + np.random.normal(0, 0.02, img.shape).astype(np.float32)
    return img.astype(np.float32), masks.astype(np.float32)


class SteelDataset(Dataset):
    def __init__(self, ids, train=False):
        self.ids, self.train = list(ids), train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        iid = self.ids[i]
        img = cv2.imread(img_paths[iid], cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        masks = np.stack([rle_decode(rle_map[(iid, c)]) for c in CLASSES]).astype(np.float32)
        if self.train:
            img, masks = augment(img, masks)
        img = (img - 0.5) / 0.5                      # same normalisation as the baseline
        return torch.from_numpy(np.ascontiguousarray(img))[None], torch.from_numpy(np.ascontiguousarray(masks))


def worker_init(worker_id):
    s = torch.initial_seed() % 2**32
    random.seed(s); np.random.seed(s)


sampler = WeightedRandomSampler(torch.from_numpy(train_w), num_samples=len(train_ids), replacement=True)
train_loader = DataLoader(SteelDataset(train_ids, True), batch_size=BATCH, sampler=sampler, num_workers=4,
                          pin_memory=True, drop_last=True, worker_init_fn=worker_init, persistent_workers=True)
val_loader = DataLoader(SteelDataset(val_ids), batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
print('train batches/epoch:', len(train_loader), '| val batches:', len(val_loader))

## 6. Model, loss and EMA
**Model:** U-Net whose encoder is EfficientNet-B3 (ImageNet-pretrained, first conv adapted to 1 channel). `aux_params` adds a small **classification head** on the deepest encoder features that outputs 4 logits: "is defect k present anywhere in this image?". The forward pass returns `(mask_logits, class_logits)`.

**Loss** = BCE(mask) + Dice(mask) + `cls_weight` x BCE(class presence).
**EMA** keeps a slowly-moving average copy of the weights; we evaluate and save the EMA copy.

In [ ]:
AUX = dict(classes=4, pooling='avg', dropout=0.2)


def build_model(encoder, pretrained=True):
    return smp.Unet(encoder_name=encoder, encoder_weights='imagenet' if pretrained else None,
                    in_channels=1, classes=4, aux_params=AUX)


ENCODER_USED = None
for enc in [CFG['encoder']] + CFG['fallback_encoders']:
    try:
        model = build_model(enc, pretrained=True)
        ENCODER_USED = enc
        print('loaded pretrained encoder:', enc)
        break
    except Exception as e:
        print('could not load', enc, '->', repr(e)[:200])
assert ENCODER_USED is not None, 'no encoder could be loaded (is Internet ON?)'
model = model.to(device)
print('parameters (M):', round(sum(p.numel() for p in model.parameters()) / 1e6, 2))

bce = nn.BCEWithLogitsLoss()
dice = smp.losses.DiceLoss(mode='multilabel', from_logits=True)   # per-class Dice over the batch; classes absent in a batch are skipped


def compute_loss(seg_logits, cls_logits, y):
    cls_target = (y.sum((2, 3)) > 0).float()
    l_seg = bce(seg_logits, y) + dice(seg_logits, y)
    l_cls = bce(cls_logits, cls_target)
    return l_seg + CFG['cls_weight'] * l_cls, l_seg.detach(), l_cls.detach()


class EMA:
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.n = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))   # faster warm-up of the average
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


ema = EMA(model, CFG['ema_decay'])

## 7. Metric and evaluation
Same competition-score approximation as the baseline: `0.7 x foreground Dice + 0.2 x (1 - false-positive area fraction) + 0.1 x empty accuracy`. During training we also report a pixel-level **macro F1** (per-class F1 over all validation pixels, averaged) because the project guideline asks for F1-style comparison across runs.

In [ ]:
FLIPS = [(), (3,), (2,), (2, 3)]


@torch.no_grad()
def predict(model, x, tta=False):
    # returns (mask probabilities (B,4,H,W), class-presence probabilities (B,4))
    seg, cls = 0, 0
    flips = FLIPS if tta else FLIPS[:1]
    with torch.amp.autocast(device_type=device, enabled=use_amp):
        for d in flips:
            s, c = model(x.flip(d) if d else x)
            s = torch.sigmoid(s.float())
            seg = seg + (s.flip(d) if d else s)
            cls = cls + torch.sigmoid(c.float())
    return seg / len(flips), cls / len(flips)


@torch.no_grad()
def evaluate(model, loader, seg_t=0.5, cls_t=0.5):
    model.eval()
    D, GA, PA, IN = [], [], [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        prob, cprob = predict(model, x)
        pred = (prob > seg_t).float() * (cprob > cls_t).float()[:, :, None, None]
        ga, pa, inter = y.sum((2, 3)), pred.sum((2, 3)), (pred * y).sum((2, 3))
        den = pa + ga
        d = torch.where(den > 0, 2 * inter / den.clamp(min=1), torch.ones_like(den))
        for lst, t in ((D, d), (GA, ga), (PA, pa), (IN, inter)):
            lst.append(t.cpu().numpy())
    return tuple(np.concatenate(v) for v in (D, GA, PA, IN))


def summarize(d, ga, pa, inter):
    is_pos = ga > 0
    is_empty = ~is_pos
    fg = float(d[is_pos].mean()) if is_pos.any() else 0.0
    fps = 1 - float((pa[is_empty] / N).mean()) if is_empty.any() else 1.0
    ea = float((pa[is_empty] == 0).mean()) if is_empty.any() else 1.0
    f1_per_class = 2 * inter.sum(0) / np.maximum(pa.sum(0) + ga.sum(0), 1)
    return {'fp_rate': float((pa[is_empty] > 0).mean()), 'foreground_dice': fg,
            'naive_mean_dice': float(d.mean()), 'macro_f1': float(f1_per_class.mean()),
            'official_score': 0.7 * fg + 0.2 * fps + 0.1 * ea}

## 8. Training
- Two learning rates: gentle for the pretrained encoder, larger for the freshly initialised decoder and heads.
- Linear warm-up (2 epochs) then cosine decay, stepped **per batch**.
- AMP (mixed precision) + gradient clipping.
- Validation uses the **EMA** model every epoch; the best EMA weights are saved; early stopping after `patience` epochs without improvement.
- `last.pt` is saved every epoch; set `CFG['resume']=True` to continue after a Kaggle session dies.

In [ ]:
enc_ids = {id(p) for p in model.encoder.parameters()}
other_params = [p for p in model.parameters() if id(p) not in enc_ids]
opt = torch.optim.AdamW([
    {'params': list(model.encoder.parameters()), 'lr': CFG['lr_encoder']},
    {'params': other_params, 'lr': CFG['lr_decoder']},
], weight_decay=CFG['weight_decay'])

steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch
warm_steps = CFG['warmup_epochs'] * steps_per_epoch


def lr_factor(step):
    if step < warm_steps:
        return 0.05 + 0.95 * step / max(1, warm_steps)
    p = (step - warm_steps) / max(1, total_steps - warm_steps)
    return CFG['min_lr_factor'] + (1 - CFG['min_lr_factor']) * 0.5 * (1 + math.cos(math.pi * p))


sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lr_factor)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

BEST_PATH, LAST_PATH = '/kaggle/working/v2_model.pt', '/kaggle/working/v2_last.pt'
best, bad, start_epoch = -1.0, 0, 0
if CFG['resume'] and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device)
    model.load_state_dict(ck['model']); ema.module.load_state_dict(ck['ema']); ema.n = ck['ema_n']
    opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
    start_epoch, best, bad = ck['epoch'], ck['best'], ck['bad']
    print('resumed from epoch', start_epoch)

wandb_config = dict(CFG, encoder_used=ENCODER_USED, epochs=EPOCHS, n_train=len(train_ids), n_val=len(val_ids),
                    loss='BCE+Dice+0.5*ClsBCE', augmentation='flips+brightness/contrast+shift/scale+noise',
                    sampler='class-weighted', scheduler='warmup+cosine (per step)', ema=True,
                    split='stratified 80/20 on label combination (same as baseline)')
try:
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=f'{RUN_NAME}_e{EPOCHS}', group='surfaceguard-segmentation',
                     tags=['unet', ENCODER_USED, 'pretrained', 'cls-head', 'ema'], config=wandb_config, mode=WANDB_MODE)
except Exception as e:
    print('wandb.init failed, logging disabled:', repr(e)[:200])
    run = wandb.init(mode='disabled')
wandb.define_metric('epoch')
wandb.define_metric('train/*', step_metric='epoch')
wandb.define_metric('val/*', step_metric='epoch')

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    model.train()
    tot = seg_l = cls_l = 0.0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device, enabled=use_amp):
            seg_logits, cls_logits = model(x)
        loss, ls, lc = compute_loss(seg_logits.float(), cls_logits.float(), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(opt)
        scaler.update()
        sched.step()
        ema.update(model)
        tot += loss.item(); seg_l += ls.item(); cls_l += lc.item()
    nb_ = len(train_loader)

    s = summarize(*evaluate(ema.module, val_loader))
    improved = s['official_score'] > best + 1e-4
    if improved:
        best, bad = s['official_score'], 0
        torch.save(ema.module.state_dict(), BEST_PATH)
    else:
        bad += 1
    torch.save(dict(model=model.state_dict(), ema=ema.module.state_dict(), ema_n=ema.n, opt=opt.state_dict(),
                    sched=sched.state_dict(), scaler=scaler.state_dict(), epoch=epoch + 1, best=best, bad=bad), LAST_PATH)

    wandb.log({'epoch': epoch + 1, 'train/loss': tot / nb_, 'train/seg_loss': seg_l / nb_, 'train/cls_loss': cls_l / nb_,
               'val/foreground_dice': s['foreground_dice'], 'val/fp_rate': s['fp_rate'], 'val/naive_mean_dice': s['naive_mean_dice'],
               'val/macro_f1': s['macro_f1'], 'val/official_score': s['official_score'],
               'lr_encoder': opt.param_groups[0]['lr'], 'lr_decoder': opt.param_groups[1]['lr'], 'time/epoch_s': time.time() - t0})
    print(f"epoch {epoch + 1}/{EPOCHS} | loss {tot / nb_:.4f} | fg_dice {s['foreground_dice']:.3f} | fp_rate {s['fp_rate']:.3f} | "
          f"macroF1 {s['macro_f1']:.3f} | official {s['official_score']:.4f} {'*' if improved else ''} | {time.time() - t0:.0f}s")
    if bad >= CFG['patience']:
        print(f'early stopping: no improvement for {bad} epochs')
        break

wandb.summary['best_val_official_score_untuned'] = best
print('best val official score (tau=0.5, cls gate 0.5, no filter):', round(best, 4))

## 9. Post-processing tuning (per class) with 4-flip TTA
For every class we search three knobs on the validation set and keep the combination that maximises that class's contribution to the competition score:
- `tau`: pixel probability threshold
- `cls_tau`: classification-head gate (if the head says "not present" with probability below this, the whole mask is dropped; `0.0` = gate off)
- `min_area`: connected regions smaller than this many pixels are deleted

Because the score is additive over classes, each class can be tuned independently. Validation probabilities are cached in memory (about 4 GB for the full split).

In [ ]:
ema.module.load_state_dict(torch.load(BEST_PATH, map_location=device))
best_model = ema.module.eval()

n_val = len(val_ids)
P = np.zeros((n_val, 4, H, W), dtype=np.float16)
Cp = np.zeros((n_val, 4), dtype=np.float32)
Y = np.zeros((n_val, 4, H, W), dtype=np.uint8)
i0 = 0
with torch.no_grad():
    for x, y in val_loader:
        prob, cprob = predict(best_model, x.to(device), tta=True)
        b = x.size(0)
        P[i0:i0 + b] = prob.half().cpu().numpy(); Cp[i0:i0 + b] = cprob.cpu().numpy(); Y[i0:i0 + b] = y.numpy().astype(np.uint8)
        i0 += b
print('cached validation probabilities:', P.shape)

gt_any = Y.reshape(n_val, 4, -1).any(2)
n_pos_total = int(gt_any.sum())
n_empty_total = int(gt_any.size - n_pos_total)
print('positive pairs:', n_pos_total, '| empty pairs:', n_empty_total)

In [ ]:
TAUS = np.round(np.arange(0.30, 0.91, 0.05), 2)
MINAREAS = [0, 50, 100, 200, 400, 800, 1500, 2500]
CLS_TAUS = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]


def class_stats(k, gt, tau, minareas):
    # intersection and predicted area per image, for every min-area value
    inter = np.zeros((len(minareas), n_val)); pa = np.zeros((len(minareas), n_val))
    mk = P[:, k] > tau
    for i in range(n_val):
        m = mk[i]
        if not m.any():
            continue
        nlab, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
        areas = st[:, cv2.CC_STAT_AREA]
        for j, ma in enumerate(minareas):
            if ma > 0:
                keep = areas >= ma
                keep[0] = False
                mm = keep[lab]
            else:
                mm = m
            pa[j, i] = mm.sum()
            inter[j, i] = (mm & gt[i]).sum()
    return inter, pa


def class_score(k, gt_area, inter, pa, cls_tau):
    keep = np.ones(n_val, bool) if cls_tau == 0.0 else Cp[:, k] > cls_tau
    posm, emp = gt_area > 0, gt_area == 0
    d = np.where(pa + gt_area > 0, 2 * inter / np.maximum(pa + gt_area, 1), 0.0)
    s_dice = d[posm & keep].sum()
    s_fp = (pa / N)[emp & keep].sum()
    s_ok = ((~keep) | (pa == 0))[emp].sum()
    return 0.7 * s_dice / n_pos_total - 0.2 * s_fp / n_empty_total + 0.1 * s_ok / n_empty_total


best_params, per_class_best, per_class_base = {}, {}, {}
t0 = time.time()
for k, c in enumerate(CLASSES):
    gt = Y[:, k] > 0
    gt_area = gt.reshape(n_val, -1).sum(1)
    best_s, best_cfg = -1e9, None
    for tau in TAUS:
        inter, pa = class_stats(k, gt, tau, MINAREAS)
        for j, ma in enumerate(MINAREAS):
            for g in CLS_TAUS:
                sc = class_score(k, gt_area, inter[j], pa[j], g)
                if tau == 0.5 and ma == 0 and g == 0.0:
                    per_class_base[c] = sc
                if sc > best_s:
                    best_s, best_cfg = sc, (float(tau), float(g), int(ma))
    best_params[c] = dict(tau=best_cfg[0], cls_tau=best_cfg[1], min_area=best_cfg[2])
    per_class_best[c] = best_s
    print(f'{c} | best {best_params[c]} | contribution tuned {best_s:.4f} vs untuned {per_class_base[c]:.4f} | {time.time() - t0:.0f}s')

val_untuned = 0.2 + sum(per_class_base.values())
val_tuned = 0.2 + sum(per_class_best.values())
print('\nvalidation score, tau=0.5, no gate, no filter (TTA):', round(val_untuned, 4))
print('validation score, tuned per class              :', round(val_tuned, 4))
print('(baseline model on the same validation split scored lower; its leaderboard score was 0.65642)')

wandb.summary['val/score_untuned_tta'] = val_untuned
wandb.summary['val/score_tuned'] = val_tuned
wandb.config.update({'tuned_params': best_params}, allow_val_change=True)

## 10. Error analysis (for the report)
Per class: how many positives, mean Dice on positives, **miss rate** (real defect, predicted empty) and **false-positive rate** (empty pair, but something predicted), plus pixel-level F1. Then a picture of predictions vs ground truth on the worst validation examples.

In [ ]:
rows = []
for k, c in enumerate(CLASSES):
    p = best_params[c]
    gt = Y[:, k] > 0
    gt_area = gt.reshape(n_val, -1).sum(1)
    inter, pa = class_stats(k, gt, p['tau'], [p['min_area']])
    inter, pa = inter[0], pa[0]
    keep = np.ones(n_val, bool) if p['cls_tau'] == 0.0 else Cp[:, k] > p['cls_tau']
    inter, pa = inter * keep, pa * keep
    posm, emp = gt_area > 0, gt_area == 0
    d = np.where(pa + gt_area > 0, 2 * inter / np.maximum(pa + gt_area, 1), 0.0)
    rows.append(dict(defect=c, positives=int(posm.sum()), fg_dice=round(float(d[posm].mean()), 4),
                     miss_rate=round(float((pa[posm] == 0).mean()), 4), fp_rate=round(float((pa[emp] > 0).mean()), 4),
                     pixel_f1=round(float(2 * inter.sum() / max(pa.sum() + gt_area.sum(), 1)), 4)))
err_table = pd.DataFrame(rows)
print(err_table.to_string(index=False))
print('macro pixel F1:', round(float(err_table['pixel_f1'].mean()), 4))
wandb.log({'error_analysis': wandb.Table(dataframe=err_table)})
wandb.summary['val/macro_pixel_f1_tuned'] = float(err_table['pixel_f1'].mean())

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def clean_mask(m, min_area):
    if min_area <= 0 or not m.any():
        return m
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    keep = st[:, cv2.CC_STAT_AREA] >= min_area
    keep[0] = False
    return keep[lab].astype(np.uint8)


def final_masks(i):
    out = np.zeros((4, H, W), np.uint8)
    for k, c in enumerate(CLASSES):
        p = best_params[c]
        if p['cls_tau'] > 0 and Cp[i, k] <= p['cls_tau']:
            continue
        out[k] = clean_mask((P[i, k] > p['tau']).astype(np.uint8), p['min_area'])
    return out


COLORS = np.array([[1, 0, 0], [0, 1, 0], [0.2, 0.4, 1], [1, 1, 0]], dtype=np.float32)   # a=red b=green c=blue d=yellow
def colorize(masks):
    rgb = np.zeros((H, W, 3), np.float32)
    for k in range(4):
        rgb += masks[k][..., None] * COLORS[k]
    return np.clip(rgb, 0, 1)


# pick the 8 positive images with the lowest per-image dice
scores = []
for i in np.where(gt_any.any(1))[0]:
    fm = final_masks(i)
    inter = (fm & Y[i]).sum(); den = fm.sum() + Y[i].sum()
    scores.append((2 * inter / max(den, 1), i))
worst = [i for _, i in sorted(scores)[:8]]

fig, axes = plt.subplots(len(worst), 3, figsize=(15, 2.2 * len(worst)))
vds = SteelDataset(val_ids)
for r, i in enumerate(worst):
    img = vds[i][0][0].numpy() * 0.5 + 0.5
    axes[r, 0].imshow(img, cmap='gray'); axes[r, 0].set_title(f'{val_ids[i]} image', fontsize=8)
    axes[r, 1].imshow(colorize(Y[i])); axes[r, 1].set_title('ground truth (a red, b green, c blue, d yellow)', fontsize=8)
    axes[r, 2].imshow(colorize(final_masks(i))); axes[r, 2].set_title('prediction', fontsize=8)
    for a in axes[r]: a.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/error_analysis.png', dpi=110)
plt.close(fig)
try:
    wandb.log({'worst_val_examples': wandb.Image('/kaggle/working/error_analysis.png')})
except Exception as e:
    print('image log skipped:', repr(e)[:120])
print('saved /kaggle/working/error_analysis.png')

## 11. Save everything needed for inference
`v2_config.json` records how to rebuild the architecture, so the inference notebook never has to guess.

In [ ]:
cfg_out = dict(arch='Unet', encoder=ENCODER_USED, in_channels=1, classes=4, aux_params=AUX, height=H, width=W,
               normalize_mean=0.5, normalize_std=0.5, classes_order=CLASSES, tta='4-flip',
               val_score_untuned=val_untuned, val_score_tuned=val_tuned)
json.dump(cfg_out, open('/kaggle/working/v2_config.json', 'w'), indent=2)
json.dump(best_params, open('/kaggle/working/v2_params.json', 'w'), indent=2)

try:
    art = wandb.Artifact('unet_v2_best', type='model')
    for f in ['v2_model.pt', 'v2_params.json', 'v2_config.json']:
        art.add_file('/kaggle/working/' + f)
    run.log_artifact(art)
except Exception as e:
    print('artifact upload skipped:', repr(e)[:150])

wandb.finish()
for f in ['v2_model.pt', 'v2_params.json', 'v2_config.json']:
    print(f, os.path.getsize('/kaggle/working/' + f), 'bytes')
print('Next: Save Version -> Save & Run All, then make a Dataset from v2_model.pt, v2_params.json, v2_config.json and smp_wheels/.')